In [1]:
# Ô 0 — môi trường. Qwen2.5-VL-7B-Instruct, KHÔNG adapter, cùng câu nhắc với Base/S1 (mốc so bên ngoài, report/322)
import subprocess, sys, os, time
T0 = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh(f"{sys.executable} -m pip install -q -U transformers peft accelerate pycocoevalcap 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y torchao 2>&1 | tail -1")
import torch
print("GPU:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], flush=True)
assert torch.cuda.device_count() == 2, "DỪNG: cần T4×2 (7B fp16 ~15,5 GB trọng số, chia hai card)"
sh(f'{sys.executable} -c "import transformers, torch; print(transformers.__version__, torch.__version__)"')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.



  Successfully uninstalled torchao-0.10.0



GPU: 2 ['Tesla T4', 'Tesla T4']


5.20.2 2.11.0+cu128



0

In [2]:
# Ô 1 — tìm dữ liệu, dựng thư mục chạy
import glob, json, shutil, hashlib
W = "/kaggle/working"
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
SC = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/only_tap.jsonl", recursive=True)]
assert len(SC) == 1, f"DỪNG: cần đúng một qwen7b-infer-script, thấy {SC}"
SC = SC[0]
assert md5(f"{SC}/infer_branch.py") == "0eef768c215ce3c3c1fc379e5a46ec32", "DỪNG: infer_branch.py lệch máy nhà"
assert md5(f"{SC}/build_branch_data.py") == "619e63e123a6dbf60086e65ee94a3912", "DỪNG: build_branch_data.py lệch máy nhà"
assert md5(f"{SC}/only_tap.jsonl") == "7480ae6da32d0bdf806af6e0d8836714"
INF = f"{W}/inf"; os.makedirs(INF, exist_ok=True)
for f in ("infer_branch.py", "build_branch_data.py"): shutil.copy(f"{SC}/{f}", f"{INF}/{f}")

TA = None
for t in glob.glob("/kaggle/input/**/test_ac/test.jsonl", recursive=True):
    root = os.path.dirname(t)
    if sum(1 for _ in open(t)) == 6958 and len(glob.glob(f"{root}/images/*.png")) >= 4463 and os.path.exists(f"{root}/ocr.jsonl"):
        TA = root
assert TA, "DỪNG: không thấy test_ac đủ 6.958 dòng + 4.463 ảnh"
assert md5(f"{TA}/test.jsonl") == "da58299ee551a926a21cbecb5232bf78", "DỪNG: test.jsonl lệch máy nhà"
PKG = os.path.dirname(os.path.dirname(os.path.dirname(TA)))
shutil.copytree(f"{PKG}/harness", f"{W}/thesis/harness", dirs_exist_ok=True, ignore=shutil.ignore_patterns("images"))
link = f"{W}/thesis/harness/dg1_cache/test_ac/images"
if os.path.lexists(link): os.remove(link)
os.symlink(f"{TA}/images", link)
WS = f"{W}/thesis"

# nối tiếp: commit 2 gắn output commit 1 làm kernel_sources → chép tệp dở sang, infer_branch tự bỏ bước đã có
PRED = f"{W}/pred_qwen7b_test.jsonl"
cu = [p for p in glob.glob("/kaggle/input/**/pred_qwen7b_test.jsonl", recursive=True)]
if cu and not os.path.exists(PRED):
    cu = max(cu, key=lambda p: sum(1 for _ in open(p)))
    shutil.copy(cu, PRED); print("nối tiếp từ", cu, sum(1 for _ in open(PRED)), "dòng", flush=True)
print("SC", SC); print("TA", TA); print("WS", WS, flush=True)

SC /kaggle/input/datasets/trangphngngc/qwen7b-infer-script
TA /kaggle/input/datasets/trangphngngc/thesis-score/thesis/harness/dg1_cache/test_ac
WS /kaggle/working/thesis


In [3]:
# Ô 2 — chạy tiến trình con, ghi log ra tệp, nhịp sống 2 phút
def chay(cmd, log, cwd, nhip=120, het_gio=None, env_them=None):
    t0 = time.time()
    with open(log, "a") as f:
        q = subprocess.Popen(cmd, cwd=cwd, stdout=f, stderr=subprocess.STDOUT,
                             env={**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
                                  "PYTHONUNBUFFERED": "1", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
                                  **(env_them or {})})
        while q.poll() is None:
            time.sleep(nhip)
            L = [l.strip() for l in open(log, errors="ignore") if l.strip()]
            print(f"  {(time.time()-T0)/3600:5.2f} h tổng · {os.path.basename(log)} · {L[-1][:110] if L else 'chưa có dòng nào'}", flush=True)
            if het_gio and time.time() - T0 > het_gio * 3600:
                q.terminate(); q.wait(); print(f"⛔ cắt ở {het_gio} h tổng để còn giờ zip", flush=True); break
    print(f"--- {os.path.basename(log)} · mã thoát {q.returncode} · {(time.time()-t0)/3600:.2f} h ---")
    print("".join(open(log, errors="ignore").readlines()[-8:]), flush=True)
    return q.returncode

ARGS = ["python", "infer_branch.py", "--no-adapter", "--base", "Qwen/Qwen2.5-VL-7B-Instruct",
        "--data-root", TA, "--recs-file", "test.jsonl", "--only", f"{SC}/only_tap.jsonl", "--batch", "4"]

In [4]:
# Ô 3 — thử 8 bước: bắt lỗi fp16 (câu rỗng/rác) và đo tốc độ TRƯỚC lượt dài
THU = f"{W}/thu_qwen7b.jsonl"
if os.path.exists(THU): os.remove(THU)
t = time.time()
chay(ARGS + ["--limit", "8", "--out", THU], f"{W}/thu.log", INF, nhip=60)
R = [json.loads(l) for l in open(THU)]
for r in R: print("·", r["pred"][:120], "|| chuẩn:", r["gold_instruction"][:60])
tot = sum(1 for r in R if len(r["pred"]) >= 5 and any(ch.isalpha() for ch in r["pred"]))
print(f"[thử] {len(R)} câu, {tot} câu có chữ, {time.time()-t:.0f} s cả nạp mô hình", flush=True)
assert len(R) == 8 and tot >= 7, "DỪNG: câu rỗng/rác ở fp16 — gửi log về, đừng chạy lượt dài"

   0.03 h tổng · thu.log · Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate l


   0.05 h tổng · thu.log · Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate l


   0.07 h tổng · thu.log · Xong 8 bước → /kaggle/working/thu_qwen7b.jsonl


--- thu.log · mã thoát 0 · 0.05 h ---
[dữ liệu] /kaggle/input/datasets/trangphngngc/thesis-score/thesis/harness/dg1_cache/test_ac/test.jsonl
khối ứng viên: KHÔNG dùng ← nhánh giữ 24 dòng OCR (S1 · S2 · MIN-DESC · CE2 · gui_s1_match)
--only: 6958 -> 4463 bước (danh sách có 4463 khoá)
Nạp Qwen/Qwen2.5-VL-7B-Instruct (mô hình gốc)
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
Xong 8 bước → /kaggle/working/thu_qwen7b.jsonl



· Tap on the video titled "Dimitri Vegas, Martin Garrix, Like Mike - Tremor" to play it. || chuẩn: click on the Dimitri vegas song
· Tap on the search bar at the top to enter "Romanticism" and find related artworks. || chuẩn: Click on the search icon 
· Press the green magnifying glass icon to search for Romanticism art. || chuẩn: Click on the search icon at the bottom right corner of the s
· Type "Venice" in the search bar and press the search icon. || chuẩn: Click on the search box
· Click on "All places in Venice, Italy" to view nearby attractions. || chuẩn: Click on All places in Venice Italy
· Tap on the "Flights" icon to start searching for flights. || chuẩn: Open the Flights Tab.
· Tap on "ONE-WAY" to switch the trip type and then tap on the departure city field to change it to Toronto Pearson Intern || chuẩn: Click on the tab One- way.
· Tap on the departure city field and change it to "Toronto Pearson International." || chuẩn: Click on the First Input box.
[thử] 8 câu, 8 câu c

In [5]:
# Ô 4 — lượt đủ 4.463 bước chạm (nối tiếp được). Cắt ở 11,0 h tổng để còn giờ zip.
chay(ARGS + ["--out", PRED], f"{W}/infer.log", INF, het_gio=11.0)
n = sum(1 for _ in open(PRED)) if os.path.exists(PRED) else 0
print(f"[qwen7b] {n}/4463 câu · {(time.time()-T0)/3600:.2f} h tổng", flush=True)
XONG = (n == 4463)

   0.10 h tổng · infer.log · warnings.warn(


   0.13 h tổng · infer.log · warnings.warn(


   0.17 h tổng · infer.log · 80/4463  (0.4 bước/giây, còn ~193 phút)


   0.20 h tổng · infer.log · 160/4463  (0.4 bước/giây, còn ~191 phút)


   0.23 h tổng · infer.log · 160/4463  (0.4 bước/giây, còn ~191 phút)


...

   3.40 h tổng · infer.log · 4320/4463  (0.4 bước/giây, còn ~6 phút)


   3.43 h tổng · infer.log · 4400/4463  (0.4 bước/giây, còn ~3 phút)


   3.47 h tổng · infer.log · Xong 4463 bước → /kaggle/working/pred_qwen7b_test.jsonl


--- infer.log · mã thoát 0 · 3.40 h ---
  3920/4463  (0.4 bước/giây, còn ~25 phút)
  4000/4463  (0.4 bước/giây, còn ~21 phút)
  4080/4463  (0.4 bước/giây, còn ~17 phút)
  4160/4463  (0.4 bước/giây, còn ~14 phút)
  4240/4463  (0.4 bước/giây, còn ~10 phút)
  4320/4463  (0.4 bước/giây, còn ~6 phút)
  4400/4463  (0.4 bước/giây, còn ~3 phút)
Xong 4463 bước → /kaggle/working/pred_qwen7b_test.jsonl



[qwen7b] 4463/4463 câu · 3.47 h tổng


In [6]:
# Ô 5 — chấm UGround nếu còn giờ (cần ~2,5–3 h [suy]); không đủ thì để commit sau (gắn output này làm kernel_sources)
CON = 11.3 - (time.time() - T0) / 3600
print(f"còn {CON:.2f} h", flush=True)
if XONG and CON >= 3.2:
    SCO = f"{W}/score_qwen7b_test.json"
    chay(["python", "-u", "harness/score_run.py", "--mode", "score", "--grounder", "uground",
          "--preds", PRED, "--out", SCO], f"{W}/cham.log", WS, het_gio=11.3, env_them={"CUDA_VISIBLE_DEVICES": "0"})
    raw = SCO.replace(".json", "_raw.jsonl")
    if os.path.exists(raw):
        R = [json.loads(l) for l in open(raw)]
        print(f"== qwen7b test: {len(R)} bước · exec {sum(int(r.get('executable', 0)) for r in R)}", flush=True)
else:
    print("BỎ QUA chấm ở commit này (chưa đủ câu hoặc thiếu giờ).", flush=True)

còn 7.83 h


   3.50 h tổng · cham.log · Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate l


   3.53 h tổng · cham.log · [14:00:24] 40/4463 =  0.9% · 0.23 bước/giây · còn ~324 phút


   3.57 h tổng · cham.log · [14:01:50] 60/4463 =  1.3% · 0.23 bước/giây · còn ~320 phút


   3.60 h tổng · cham.log · [14:04:41] 100/4463 =  2.2% · 0.23 bước/giây · còn ~315 phút


...

   8.80 h tổng · cham.log · [19:16:03] 4420/4463 = 99.0% · 0.23 bước/giây · còn ~3 phút


   8.83 h tổng · cham.log · [19:17:30] 4440/4463 = 99.5% · 0.23 bước/giây · còn ~2 phút


   8.87 h tổng · cham.log · /kaggle/working/score_qwen7b_test_raw.jsonl  (4463 bước — đổi luật chấm hay thêm lát cắt thì chấm lại từ tệp n


--- cham.log · mã thoát 0 · 5.40 h ---
CHẤM — pred_qwen7b_test.jsonl  ·  4463 bước chạm
  ô-Voronoi tâm (headline):  51.5%   KTC95 [49.8%, 53.1%]
  đĩa dung sai (báo kèm) :  61.2%   KTC95 [59.5%, 62.9%]
  cụm: 1091 (hiệu dụng 454.3)

Đã lưu /kaggle/working/score_qwen7b_test.json
         /kaggle/working/score_qwen7b_test_raw.jsonl  (4463 bước — đổi luật chấm hay thêm lát cắt thì chấm lại từ tệp này, KHÔNG gọi lại bộ trỏ)



== qwen7b test: 4463 bước · exec 2299


In [ ]:
# Ô 6 — gói kết quả thành một zip
import zipfile
fs = [f for f in glob.glob(f"{W}/*") if os.path.isfile(f) and not f.endswith(".zip") and os.path.getsize(f) < 200*2**20]
with zipfile.ZipFile(f"{W}/qwen7b_test.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in fs: z.write(f, os.path.basename(f))
print("đã zip", len(fs), "tệp:", sorted(os.path.basename(f) for f in fs), flush=True)